# MTU Notebook Workflow

This notebook demonstrates how to run Mapbox Tileset Uploader from Python in a shareable format.

Default behavior is **dry run** so you can validate conversion and geometry checks without uploading.

## Quick Order (run top to bottom)

1. Set runtime options and credentials in Cell 3.
2. Review limits/disclaimers and supported formats.
3. Run preflight checks.
4. Run upload cell (dry run first, then real upload).

## Environment Options

1. Custom clean venv (recommended for packaging and general notebook work).
2. ArcGIS Pro cloned conda environment (recommended if you rely on ArcGIS/geospatial stack compatibility).

Use one environment option, then run the notebook cells below.

In [ ]:
# pyright: reportMissingImports=false, reportUnknownVariableType=false, reportUnknownMemberType=false, reportUnknownArgumentType=false, reportUnnecessaryComparison=false
# Environment setup options (run these in terminal, not necessarily in this cell).

# Option A: custom clean venv
# py -3.11 -m venv .venv-notebook
# .\.venv-notebook\Scripts\python.exe -m pip install --upgrade pip
# .\.venv-notebook\Scripts\python.exe -m pip install -e ".[all]"

# Option B: ArcGIS Pro cloned conda env (example names/paths)
# conda create --name arcgispro-mtu --clone arcgispro-py3
# conda activate arcgispro-mtu
# python -m pip install --upgrade pip
# python -m pip install -e ".[all]"

# If you are only consuming from PyPI instead of source:
# %pip install mtu

In [ ]:
# Runtime configuration (edit these first).
import os
from pathlib import Path

# Credentials (prefer environment variables; do not hardcode secrets)
MAPBOX_ACCESS_TOKEN = os.environ.get('MAPBOX_ACCESS_TOKEN', '')
MAPBOX_USERNAME = os.environ.get('MAPBOX_USERNAME', '')

DRY_RUN = True

# Choose one source mode: 'file' or 'url'
SOURCE_MODE: str = os.environ.get('MTU_SOURCE_MODE', 'file').strip().lower()
SOURCE_FILE = Path('.../data.geojson')
SOURCE_URL = 'https://example.com/data.geojson'

# Optional format hint (set to None for auto-detect).
# Examples: 'GeoJSON', 'TopoJSON', 'Shapefile', 'GeoPackage', 'KML/KMZ',
# 'FlatGeobuf', 'GeoParquet', 'GPX'
FORMAT_HINT = None

TILESET_ID = 'demo-tileset-id'
TILESET_NAME = 'Demo Tileset'
LAYER_NAME = 'data'
MIN_ZOOM = 4
MAX_ZOOM = 8

# Mapbox limits and app guardrails
MAPBOX_ZOOM_MIN = 0
MAPBOX_ZOOM_MAX = 22
APP_DEFAULT_UPLOAD_CAP_GB = 1
MAPBOX_FULL_UPLOAD_CAP_GB = 20
USE_MAPBOX_FULL_UPLOAD_CAP = False

# Optional capacity planning fields (set to 0 to ignore)
CAPACITY_LIMIT_MB = 0.0
CAPACITY_USED_MB = 0.0

if SOURCE_MODE not in {'file', 'url'}:
    raise ValueError("SOURCE_MODE must be 'file' or 'url'")

token_set = bool(MAPBOX_ACCESS_TOKEN)
username_set = bool(MAPBOX_USERNAME)

print(f'DRY_RUN: {DRY_RUN}')
print(f'SOURCE_MODE: {SOURCE_MODE}')
print(f'SOURCE_FILE: {SOURCE_FILE}')
print(f'SOURCE_URL: {SOURCE_URL}')
print(f'FORMAT_HINT: {FORMAT_HINT}')
print(f'Zoom: {MIN_ZOOM}-{MAX_ZOOM} (Mapbox typical range {MAPBOX_ZOOM_MIN}-{MAPBOX_ZOOM_MAX})')
print('Upload cap mode:', 'Mapbox full 20 GB' if USE_MAPBOX_FULL_UPLOAD_CAP else 'App default 1 GB')
print(f'MAPBOX_ACCESS_TOKEN set: {token_set}')
print(f'MAPBOX_USERNAME set: {username_set}')

DRY_RUN: True
SOURCE_MODE: file
SOURCE_FILE: C:\Users\ayiem\United Nations\OCHA ROSEA - IM Unit\GIS Data\2020\3_Countries\Madagascar\MDG_COD_AB\mdg_adm_bngrc_ocha_20181031_shp\mdg_admbnda_adm2_BNGRC_OCHA_20181031.zip
SOURCE_URL: https://example.com/data.geojson
FORMAT_HINT: None
Zoom: 4-8 (Mapbox typical range 0-22)
Upload cap mode: App default 1 GB


## Mapbox Limits, Input Types, and Disclaimers

### Supported input data types

- GeoJSON (`.geojson`, `.json`)
- TopoJSON (`.topojson`)
- Shapefile (`.shp`, `.zip`)
- GeoPackage (`.gpkg`)
- KML/KMZ (`.kml`, `.kmz`)
- FlatGeobuf (`.fgb`)
- GeoParquet (`.parquet`, `.geoparquet`)
- GPX (`.gpx`)

### Source mode

- Use `SOURCE_MODE = 'file'` with `SOURCE_FILE` for local inputs.
- Use `SOURCE_MODE = 'url'` with `SOURCE_URL` for remote inputs.
- Leave `FORMAT_HINT = None` for automatic detection unless you need to force a format.

### Limits and warnings

- Required token scopes: `tilesets:read`, `tilesets:write`, `tilesets:list`.
- URL-restricted tokens may fail with `403 Forbidden` in CLI/notebook flows.
- Mapbox zoom range is typically `0-22`; notebook default is `4-8`.
- This workflow defaults to a **1 GB upload cap** for safety.
- You can opt into Mapbox's **20 GB per-file cap** with `USE_MAPBOX_FULL_UPLOAD_CAP = True`.
- Remaining Mapbox CU cannot be queried directly from this notebook.
- Higher zoom ranges and heavier datasets can increase billable usage.
- Review your Mapbox plan limits and pricing before large uploads.

In [ ]:
import json
import os
from typing import Any

from mtu.uploader import TilesetConfig, TilesetUploader

In [ ]:
# pyright: reportMissingImports=false, reportUnknownVariableType=false, reportUnknownMemberType=false, reportUnknownArgumentType=false, reportUnnecessaryComparison=false
"""Preflight checks + optional map preview (GeoJSON local file mode only)."""

cap_gb = MAPBOX_FULL_UPLOAD_CAP_GB if USE_MAPBOX_FULL_UPLOAD_CAP else APP_DEFAULT_UPLOAD_CAP_GB
cap_mb = cap_gb * 1024
print(f'Active upload cap: {cap_gb} GB')

if SOURCE_MODE == 'url':
    print(f'URL source selected: {SOURCE_URL}')
    if CAPACITY_LIMIT_MB > 0:
        print('Note: local size/capacity projection is skipped for URL sources.')
else:
    source_file = SOURCE_FILE
    if not source_file.exists():
        print(f'WARNING: file not found: {source_file}')
    else:
        size_mb = source_file.stat().st_size / (1024 * 1024)
        print(f'Input size: {size_mb:.2f} MB')
        if size_mb > cap_mb:
            print('WARNING: input file exceeds active upload cap.')
        if CAPACITY_LIMIT_MB > 0:
            projected = CAPACITY_USED_MB + size_mb
            print(f'Projected usage: {projected:.2f} / {CAPACITY_LIMIT_MB:.2f} MB')
            if projected > CAPACITY_LIMIT_MB:
                print('WARNING: projected usage exceeds configured capacity limit.')

        # Optional map preview for local GeoJSON files using folium.
        if source_file.suffix.lower() in {'.geojson', '.json'}:
            try:
                import folium
                from IPython.display import display

                with source_file.open('r', encoding='utf-8') as f:
                    gj: Any = json.load(f)

                m = folium.Map(location=[0, 0], zoom_start=MIN_ZOOM, control_scale=True)
                layer = folium.GeoJson(gj, name='source data')
                layer.add_to(m)

                try:
                    b = layer.get_bounds()
                    if b and len(b) == 2:
                        sw = b[0]
                        ne = b[1]
                        if (
                            sw[0] is not None
                            and sw[1] is not None
                            and ne[0] is not None
                            and ne[1] is not None
                        ):
                            bounds = [[float(sw[0]), float(sw[1])], [float(ne[0]), float(ne[1])]]
                            m.fit_bounds(bounds)
                except Exception:
                    pass

                display(m)
            except Exception as ex:
                print('Map preview skipped (folium not installed or invalid GeoJSON):', ex)

Active upload cap: 1 GB
Input size: 7.38 MB


In [6]:
# Uses runtime configuration from the earlier config cell.
config = TilesetConfig(
    tileset_id=TILESET_ID,
    tileset_name=TILESET_NAME,
    layer_name=LAYER_NAME,
    min_zoom=MIN_ZOOM,
    max_zoom=MAX_ZOOM,
)

uploader = TilesetUploader(
    validate_geometry=True,
    use_mapbox_full_upload_cap=USE_MAPBOX_FULL_UPLOAD_CAP,
)

if SOURCE_MODE == 'url':
    result = uploader.upload_from_url(
        url=SOURCE_URL,
        config=config,
        format_hint=FORMAT_HINT,
        dry_run=DRY_RUN,
    )
else:
    result = uploader.upload_from_file(
        file_path=SOURCE_FILE,
        config=config,
        format_hint=FORMAT_HINT,
        dry_run=DRY_RUN,
    )

print('success:', result.success)
print('dry_run:', result.dry_run)
print('tileset_id:', result.tileset_id)
print('steps:', result.steps)
print('warnings:', result.warnings)

success: True
dry_run: True
tileset_id: ocha-rosea-1.demo-tileset-id
steps: {'convert': True, 'validate': True}
warnings: []


## Enable Real Upload

After dry-run checks pass, switch `dry_run=False` in the upload cell to publish to Mapbox.

## Sharing Tips

- Share this notebook via GitHub so others can open it in Jupyter/VS Code.
- Keep secrets out of notebook cells and outputs.
- Prefer environment variables or a local `.env` that is ignored by git.